# Eyes-closed resting EEG

Converted from `resting_state.py`. Run cells in order. The last cell starts the interactive scorer / plots.

Data live under `experiments/LevelN/Results_CSV` and `Results_xdf` (not in git). Outputs go to `analysis/outputs/`.


In [ ]:
# Run from repo root or from analysis/. This replaces __file__ used by the .py scripts.
from pathlib import Path
import os, sys
cwd = Path.cwd().resolve()
if (cwd / "analysis").is_dir() and (cwd / "experiments").is_dir():
    REPO_ROOT = cwd
elif cwd.name == "analysis":
    REPO_ROOT = cwd.parent
elif cwd.name == "notebooks":
    REPO_ROOT = cwd.parents[1]
else:
    REPO_ROOT = cwd
_ANALYSIS_DIR = REPO_ROOT / "analysis"
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
%matplotlib inline


Resting State EEG Analysis - Pre/Post Experiment Comparison
===========================================================

This script analyzes resting state EEG data collected before and after
an experiment to assess cognitive fatigue, mental state changes, and
recovery potential.

Data collection protocol:
- Eyes CLOSED resting state
- 2-3 minutes duration (minimum 1 minute recommended)
- Quiet environment, participant relaxed

Muse EEG channels:
- TP9: Left temporal
- AF7: Left frontal  
- AF8: Right frontal
- TP10: Right temporal

Metrics calculated:
1. Power Spectral Density (all frequency bands)
2. Frequency Ratios (Theta/Beta, Alpha/Beta, Theta/Alpha)
3. Individual Alpha Frequency (IAF)
4. Frontal Alpha Asymmetry (FAA)
5. 1/f Aperiodic Slope (Neural Noise)
6. Complexity Measures (Sample Entropy, Higuchi FD)
7. Pre-Post Change Analysis
8. Mental Fatigue Index

Author: Auto-generated for AttentionSpan project


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.stats import pearsonr, zscore
from scipy.fft import fft, fftfreq
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


In [ ]:
class RestingStateAnalyzer:
    """
    Comprehensive resting state EEG analyzer for pre/post experiment comparison.
    """
    
    def __init__(self, base_path=None):
        """Initialize the analyzer with base path for data files."""
        repo = os.path.abspath(os.path.join(str(_ANALYSIS_DIR), ".."))
        if base_path is None:
            self.base_path = os.environ.get(
                "ATTENTIONSPAN_DATA", os.path.join(repo, "experiments")
            )
        else:
            self.base_path = base_path
            
        # Muse EEG channels
        self.eeg_channels = ['TP9', 'AF7', 'AF8', 'TP10']
        self.frontal_channels = ['AF7', 'AF8']
        self.temporal_channels = ['TP9', 'TP10']
        
        # Frequency bands (eyes closed resting state - alpha typically stronger)
        self.frequency_bands = {
            'delta': (1, 4),
            'theta': (4, 8),
            'alpha': (8, 13),
            'beta': (13, 30),
            'gamma': (30, 45)
        }
        
        # Standard Muse sampling rate
        self.sampling_rate = 256
        
    def load_eeg_data(self, filepath):
        """
        Load EEG data from file (supports XDF as CSV or actual CSV).
        
        Args:
            filepath: Path to the EEG file
            
        Returns:
            DataFrame with EEG data, or None if loading fails
        """
        if not os.path.exists(filepath):
            print(f"Error: File not found: {filepath}")
            return None
            
        try:
            # Try loading as CSV first (many XDF files are actually CSV format)
            df = pd.read_csv(filepath)
            
            # Check for required columns
            if 'timestamps' in df.columns:
                df = df.rename(columns={'timestamps': 'timestamp'})
            
            # Verify EEG channels exist
            available_channels = [ch for ch in self.eeg_channels if ch in df.columns]
            if len(available_channels) < 2:
                print(f"Warning: Only found {len(available_channels)} EEG channels: {available_channels}")
                
            print(f"Loaded {len(df)} samples from {os.path.basename(filepath)}")
            print(f"  Duration: {len(df)/self.sampling_rate:.1f} seconds")
            print(f"  Channels: {available_channels}")
            
            return df
            
        except Exception as e:
            print(f"Error loading file: {e}")
            
            # Try loading as actual XDF
            try:
                import pyxdf
                streams, _ = pyxdf.load_xdf(filepath)
                
                for stream in streams:
                    if len(stream['time_stamps']) > 0:
                        data = stream['time_series']
                        timestamps = stream['time_stamps']
                        
                        # Create DataFrame
                        df = pd.DataFrame(data, columns=self.eeg_channels[:data.shape[1]])
                        df['timestamp'] = timestamps
                        
                        print(f"Loaded {len(df)} samples from XDF: {os.path.basename(filepath)}")
                        return df
                        
            except ImportError:
                print("pyxdf not installed - cannot load binary XDF files")
            except Exception as e2:
                print(f"Failed to load as XDF: {e2}")
                
        return None
    
    def preprocess_resting_eeg(self, eeg_data, remove_artifacts=True, artifact_threshold_uv=1500.0):
        """
        Preprocess resting state EEG data.
        
        Args:
            eeg_data: DataFrame with EEG data
            remove_artifacts: Whether to remove artifact-contaminated segments
            artifact_threshold_uv: Absolute amplitude above this (µV) marks a sample as artifact.
                Muse exports can exceed 100 µV on frontal channels; 1500 µV is a practical default.
            
        Returns:
            Preprocessed DataFrame
        """
        df = eeg_data.copy()
        
        # Get available channels
        available_channels = [ch for ch in self.eeg_channels if ch in df.columns]
        
        for channel in available_channels:
            signal_data = df[channel].values
            
            # 1. Remove DC offset
            signal_data = signal_data - np.mean(signal_data)
            
            # 2. Bandpass filter (1-45 Hz for resting state)
            nyquist = self.sampling_rate / 2
            low = 1 / nyquist
            high = 45 / nyquist
            b, a = signal.butter(4, [low, high], btype='band')
            signal_data = signal.filtfilt(b, a, signal_data)
            
            # 3. Notch filter for 60Hz line noise
            notch_freq = 60
            Q = 30
            b_notch, a_notch = signal.iirnotch(notch_freq, Q, self.sampling_rate)
            signal_data = signal.filtfilt(b_notch, a_notch, signal_data)
            
            df[channel] = signal_data
        
        if remove_artifacts:
            # Remove segments with extreme values (blinks/movement; threshold in µV)
            artifact_mask = np.zeros(len(df), dtype=bool)
            
            for channel in available_channels:
                channel_data = df[channel].values
                # Mark samples exceeding threshold
                artifact_mask |= np.abs(channel_data) > artifact_threshold_uv
            
            # Expand mask to remove 0.5s around artifacts
            expand_samples = int(0.5 * self.sampling_rate)
            expanded_mask = artifact_mask.copy()
            for i in range(len(artifact_mask)):
                if artifact_mask[i]:
                    start = max(0, i - expand_samples)
                    end = min(len(artifact_mask), i + expand_samples)
                    expanded_mask[start:end] = True
            
            # Remove artifact segments
            clean_df = df[~expanded_mask].reset_index(drop=True)
            
            removed_pct = (np.sum(expanded_mask) / len(df)) * 100
            print(f"  Artifact removal: {removed_pct:.1f}% of data removed")
            
            if len(clean_df) < self.sampling_rate * 30:  # Less than 30 seconds
                print("  Warning: Less than 30 seconds of clean data remaining")
                
            return clean_df
        
        return df
    
    def compute_power_spectral_density(self, eeg_data, channel):
        """
        Compute power spectral density using Welch's method.
        
        Args:
            eeg_data: DataFrame with EEG data
            channel: Channel name to analyze
            
        Returns:
            frequencies, psd arrays
        """
        if channel not in eeg_data.columns:
            return None, None
            
        signal_data = eeg_data[channel].values
        
        # Use Welch's method with 4-second windows (good for alpha band)
        nperseg = min(4 * self.sampling_rate, len(signal_data) // 2)
        
        frequencies, psd = signal.welch(
            signal_data,
            fs=self.sampling_rate,
            nperseg=nperseg,
            noverlap=nperseg // 2,
            window='hann'
        )
        
        return frequencies, psd
    
    def compute_band_powers(self, eeg_data):
        """
        Compute absolute and relative band powers for all channels.
        
        Returns:
            Dictionary with band powers per channel
        """
        band_powers = {}
        relative_powers = {}
        
        available_channels = [ch for ch in self.eeg_channels if ch in eeg_data.columns]
        
        for channel in available_channels:
            frequencies, psd = self.compute_power_spectral_density(eeg_data, channel)
            
            if frequencies is None:
                continue
                
            channel_powers = {}
            channel_relative = {}
            total_power = 0
            
            for band_name, (low, high) in self.frequency_bands.items():
                band_mask = (frequencies >= low) & (frequencies < high)
                band_power = (
                    np.trapezoid(psd[band_mask], frequencies[band_mask])
                    if hasattr(np, "trapezoid")
                    else np.trapz(psd[band_mask], frequencies[band_mask])
                )
                channel_powers[band_name] = band_power
                total_power += band_power
            
            # Compute relative powers
            for band_name, power in channel_powers.items():
                channel_relative[band_name] = power / total_power if total_power > 0 else 0
            
            band_powers[channel] = channel_powers
            relative_powers[channel] = channel_relative
        
        return band_powers, relative_powers
    
    def compute_individual_alpha_frequency(self, eeg_data):
        """
        Alpha peak / IAF: for each of the four Muse channels, frequency (Hz) at which
        Welch PSD is maximal in 8–13 Hz. Summary metrics use all available channels:
        overall = mean of those peaks; min/max = spread across the four channel peaks.
        """
        iaf_per_channel = {}
        
        available_channels = [ch for ch in self.eeg_channels if ch in eeg_data.columns]
        
        for channel in available_channels:
            frequencies, psd = self.compute_power_spectral_density(eeg_data, channel)
            
            if frequencies is None:
                continue
            
            # Focus on alpha band (8-13 Hz)
            alpha_mask = (frequencies >= 8) & (frequencies <= 13)
            alpha_freqs = frequencies[alpha_mask]
            alpha_psd = psd[alpha_mask]
            
            if len(alpha_psd) > 0:
                # Find peak frequency
                peak_idx = np.argmax(alpha_psd)
                iaf_per_channel[channel] = alpha_freqs[peak_idx]
            else:
                iaf_per_channel[channel] = np.nan
        
        # Primary summary: mean / min / max across the four standard channel peaks (TP9, AF7, AF8, TP10)
        peaks_four = [
            iaf_per_channel[c]
            for c in self.eeg_channels
            if c in iaf_per_channel and not np.isnan(iaf_per_channel[c])
        ]
        overall_iaf = float(np.mean(peaks_four)) if peaks_four else np.nan
        min_across_channels_hz = float(np.min(peaks_four)) if peaks_four else np.nan
        max_across_channels_hz = float(np.max(peaks_four)) if peaks_four else np.nan

        # Frontal-only mean (optional reference; fatigue pipeline uses overall = all-channel mean)
        frontal_vals = [
            iaf_per_channel[c]
            for c in self.frontal_channels
            if c in iaf_per_channel and not np.isnan(iaf_per_channel[c])
        ]
        frontal_mean_hz = float(np.mean(frontal_vals)) if frontal_vals else np.nan

        return {
            'per_channel': iaf_per_channel,
            'overall': overall_iaf,
            'min_across_channels_hz': min_across_channels_hz,
            'max_across_channels_hz': max_across_channels_hz,
            'frontal_mean_hz': frontal_mean_hz,
        }
    
    def compute_frontal_alpha_asymmetry(self, eeg_data):
        """
        Compute Frontal Alpha Asymmetry (FAA).
        
        FAA = (Right_Alpha - Left_Alpha) / (Right_Alpha + Left_Alpha)
        
        Positive FAA: Greater right alpha = left frontal activity = approach motivation
        Negative FAA: Greater left alpha = right frontal activity = withdrawal motivation
        
        Returns:
            FAA value and interpretation
        """
        band_powers, _ = self.compute_band_powers(eeg_data)
        
        # Get frontal channel alpha powers
        left_alpha = band_powers.get('AF7', {}).get('alpha', np.nan)
        right_alpha = band_powers.get('AF8', {}).get('alpha', np.nan)
        
        if np.isnan(left_alpha) or np.isnan(right_alpha):
            return {
                'faa': np.nan,
                'interpretation': 'Insufficient data',
                'left_alpha': left_alpha,
                'right_alpha': right_alpha
            }
        
        # Compute FAA (log-transformed is standard in literature)
        left_log = np.log(left_alpha + 1e-10)
        right_log = np.log(right_alpha + 1e-10)
        
        faa = right_log - left_log
        
        # Interpretation
        if faa > 0.1:
            interpretation = "Approach motivation / Positive affect"
        elif faa < -0.1:
            interpretation = "Withdrawal motivation / Negative affect"
        else:
            interpretation = "Neutral / Balanced"
        
        return {
            'faa': faa,
            'interpretation': interpretation,
            'left_alpha': left_alpha,
            'right_alpha': right_alpha,
            'left_log': left_log,
            'right_log': right_log
        }
    
    def compute_frequency_ratios(self, eeg_data):
        """
        Compute key frequency ratios indicative of cognitive state.
        
        Returns:
            Dictionary with ratios per channel and averages
        """
        band_powers, _ = self.compute_band_powers(eeg_data)
        
        ratios = {
            'theta_beta': {},      # Higher = more mind-wandering/less focus
            'alpha_beta': {},      # Higher = more relaxed
            'theta_alpha': {},     # Higher = drowsiness/fatigue
            'beta_alpha': {}       # Higher = more alert/anxious
        }
        
        available_channels = [ch for ch in self.eeg_channels if ch in band_powers]
        
        for channel in available_channels:
            powers = band_powers[channel]
            
            theta = powers.get('theta', 0)
            alpha = powers.get('alpha', 0)
            beta = powers.get('beta', 0)
            
            ratios['theta_beta'][channel] = theta / (beta + 1e-10)
            ratios['alpha_beta'][channel] = alpha / (beta + 1e-10)
            ratios['theta_alpha'][channel] = theta / (alpha + 1e-10)
            ratios['beta_alpha'][channel] = beta / (alpha + 1e-10)
        
        # Compute averages
        for ratio_name in ratios:
            values = list(ratios[ratio_name].values())
            ratios[ratio_name]['average'] = np.mean(values) if values else np.nan
        
        return ratios
    
    def compute_aperiodic_slope(self, eeg_data, channel='AF7'):
        """
        Compute the 1/f aperiodic slope of the power spectrum.
        
        The aperiodic (non-oscillatory) component reflects the balance between
        excitation and inhibition in the brain. Flatter slope = more neural noise.
        
        Returns:
            Slope value and offset
        """
        frequencies, psd = self.compute_power_spectral_density(eeg_data, channel)
        
        if frequencies is None:
            return {'slope': np.nan, 'offset': np.nan}
        
        # Fit in log-log space, excluding very low and very high frequencies
        freq_mask = (frequencies >= 2) & (frequencies <= 40)
        
        log_freq = np.log10(frequencies[freq_mask])
        log_psd = np.log10(psd[freq_mask] + 1e-10)
        
        # Linear fit
        try:
            slope, offset = np.polyfit(log_freq, log_psd, 1)
        except:
            return {'slope': np.nan, 'offset': np.nan}
        
        return {
            'slope': slope,
            'offset': offset,
            'interpretation': 'Steeper (more negative) = better E/I balance'
        }
    
    def compute_sample_entropy(self, signal_data, m=2, r=0.2):
        """
        Compute Sample Entropy - a measure of signal complexity.
        
        Higher entropy = more complex/less predictable signal.
        Decreased entropy post-experiment may indicate fatigue.
        
        Args:
            signal_data: 1D array of signal values
            m: Embedding dimension (default 2)
            r: Tolerance (default 0.2 * std)
            
        Returns:
            Sample entropy value
        """
        N = len(signal_data)
        
        if N < 100:
            return np.nan
        
        # Downsample if too long — sample entropy below is O(N^2); keep N modest.
        max_n = 600
        if N > max_n:
            step = max(1, N // max_n)
            signal_data = signal_data[::step]
            N = len(signal_data)
        
        r = r * np.std(signal_data)
        
        def _count_matches(template_length):
            count = 0
            for i in range(N - template_length):
                template = signal_data[i:i + template_length]
                for j in range(i + 1, N - template_length):
                    match = signal_data[j:j + template_length]
                    if np.max(np.abs(template - match)) <= r:
                        count += 1
            return count
        
        try:
            A = _count_matches(m + 1)
            B = _count_matches(m)
            
            if B == 0:
                return np.nan
            
            return -np.log(A / B) if A > 0 else np.nan
        except:
            return np.nan
    
    def compute_higuchi_fd(self, signal_data, kmax=10):
        """
        Compute Higuchi Fractal Dimension - another complexity measure.
        
        Higher FD = more complex signal.
        
        Args:
            signal_data: 1D array
            kmax: Maximum k value
            
        Returns:
            Fractal dimension value
        """
        N = len(signal_data)
        
        if N < kmax * 10:
            return np.nan
        
        L = []
        x = []
        
        for k in range(1, kmax + 1):
            Lk = []
            for m in range(1, k + 1):
                Lmk = 0
                indices = np.arange(m, N, k)
                
                if len(indices) < 2:
                    continue
                    
                for i in range(len(indices) - 1):
                    Lmk += abs(signal_data[indices[i + 1]] - signal_data[indices[i]])
                
                Lmk = (Lmk * (N - 1)) / (k * ((N - m) // k) * k)
                Lk.append(Lmk)
            
            if Lk:
                L.append(np.mean(Lk))
                x.append(k)
        
        if len(L) < 3:
            return np.nan
        
        try:
            slope, _ = np.polyfit(np.log(x), np.log(L), 1)
            return -slope
        except:
            return np.nan
    
    def compute_complexity_measures(self, eeg_data):
        """
        Compute complexity measures for all channels.
        
        Returns:
            Dictionary with complexity metrics per channel
        """
        complexity = {}
        
        available_channels = [ch for ch in self.eeg_channels if ch in eeg_data.columns]
        
        for channel in available_channels:
            signal_data = eeg_data[channel].values
            
            complexity[channel] = {
                'sample_entropy': self.compute_sample_entropy(signal_data),
                'higuchi_fd': self.compute_higuchi_fd(signal_data)
            }
        
        # Compute averages
        se_values = [complexity[ch]['sample_entropy'] for ch in complexity 
                     if not np.isnan(complexity[ch]['sample_entropy'])]
        hfd_values = [complexity[ch]['higuchi_fd'] for ch in complexity 
                      if not np.isnan(complexity[ch]['higuchi_fd'])]
        
        complexity['average'] = {
            'sample_entropy': np.mean(se_values) if se_values else np.nan,
            'higuchi_fd': np.mean(hfd_values) if hfd_values else np.nan
        }
        
        return complexity
    
    def compute_baseline_variability(self, eeg_data, window_seconds=10):
        """
        Compute variability (SD) of key metrics over sliding windows.
        
        This enables personalized normalization: changes are measured in
        units of "how much this person's brain normally fluctuates."
        
        Args:
            eeg_data: Baseline EEG data (DataFrame)
            window_seconds: Size of sliding window in seconds
            
        Returns:
            Dictionary with SD values for each key metric
        """
        window_samples = int(window_seconds * self.sampling_rate)
        n_samples = len(eeg_data)
        
        if n_samples < window_samples * 2:
            print(f"   Warning: Baseline too short for variability calculation")
            print(f"   Using fallback normalization values")
            return {
                'theta_beta_sd': 0.10,
                'iaf_sd': 0.3,
                'alpha_relative_sd': 5.0,
                'sample_entropy_sd': 0.05
            }
        
        # Sliding window metrics
        theta_beta_values = []
        iaf_values = []
        alpha_relative_values = []
        entropy_values = []
        
        # Slide through the data with 50% overlap (cap windows — entropy per window is costly)
        step = window_samples // 2
        starts = list(range(0, n_samples - window_samples, step))
        max_windows = 12
        if len(starts) > max_windows:
            picks = np.linspace(0, len(starts) - 1, max_windows, dtype=int)
            starts = [starts[i] for i in picks]
        n_windows = 0
        
        for start in starts:
            end = start + window_samples
            window_data = eeg_data.iloc[start:end]
            
            try:
                # Theta/Beta ratio for this window
                ratios = self.compute_frequency_ratios(window_data)
                tb = ratios['theta_beta']['average']
                if not np.isnan(tb) and tb > 0:
                    theta_beta_values.append(tb)
                
                # IAF for this window
                iaf_result = self.compute_individual_alpha_frequency(window_data)
                iaf = iaf_result['overall']
                if not np.isnan(iaf) and iaf > 0:
                    iaf_values.append(iaf)
                
                # Alpha relative power for this window
                _, relative_powers = self.compute_band_powers(window_data)
                alpha_vals = [relative_powers[ch]['alpha'] * 100 
                             for ch in relative_powers 
                             if 'alpha' in relative_powers[ch]]
                if alpha_vals:
                    alpha_relative_values.append(np.mean(alpha_vals))
                
                # Sample entropy (use smaller subset for speed)
                for channel in ['AF7', 'AF8']:
                    if channel in window_data.columns:
                        se = self.compute_sample_entropy(window_data[channel].values)
                        if not np.isnan(se):
                            entropy_values.append(se)
                        break
                
                n_windows += 1
                
            except Exception:
                continue
        
        print(f"\n   Baseline Variability Analysis ({n_windows} windows):")
        
        # Calculate SDs with fallbacks
        variability = {}
        
        if len(theta_beta_values) >= 3:
            variability['theta_beta_sd'] = np.std(theta_beta_values)
            variability['theta_beta_mean'] = np.mean(theta_beta_values)
            print(f"   θ/β ratio:      mean={variability['theta_beta_mean']:.3f}, SD={variability['theta_beta_sd']:.3f}")
        else:
            variability['theta_beta_sd'] = 0.10
            variability['theta_beta_mean'] = 0.50
            print(f"   θ/β ratio:      Using fallback SD=0.10")
        
        if len(iaf_values) >= 3:
            variability['iaf_sd'] = np.std(iaf_values)
            variability['iaf_mean'] = np.mean(iaf_values)
            print(f"   IAF:            mean={variability['iaf_mean']:.2f} Hz, SD={variability['iaf_sd']:.2f} Hz")
        else:
            variability['iaf_sd'] = 0.3
            variability['iaf_mean'] = 10.0
            print(f"   IAF:            Using fallback SD=0.3 Hz")
        
        if len(alpha_relative_values) >= 3:
            variability['alpha_relative_sd'] = np.std(alpha_relative_values)
            variability['alpha_relative_mean'] = np.mean(alpha_relative_values)
            print(f"   Alpha power:    mean={variability['alpha_relative_mean']:.1f}%, SD={variability['alpha_relative_sd']:.1f}%")
        else:
            variability['alpha_relative_sd'] = 5.0
            variability['alpha_relative_mean'] = 30.0
            print(f"   Alpha power:    Using fallback SD=5.0%")
        
        if len(entropy_values) >= 3:
            variability['sample_entropy_sd'] = np.std(entropy_values)
            variability['sample_entropy_mean'] = np.mean(entropy_values)
            print(f"   Sample Entropy: mean={variability['sample_entropy_mean']:.3f}, SD={variability['sample_entropy_sd']:.3f}")
        else:
            variability['sample_entropy_sd'] = 0.05
            variability['sample_entropy_mean'] = 0.5
            print(f"   Sample Entropy: Using fallback SD=0.05")
        
        return variability
    
    def analyze_resting_state(self, eeg_data, label=''):
        """
        Run complete resting state analysis on EEG data.
        
        Args:
            eeg_data: DataFrame with preprocessed EEG data
            label: Label for this recording (e.g., 'pre' or 'post')
            
        Returns:
            Dictionary with all computed metrics
        """
        print(f"\n{'='*60}")
        print(f"RESTING STATE ANALYSIS: {label}")
        print(f"{'='*60}")
        
        results = {
            'label': label,
            'duration_seconds': len(eeg_data) / self.sampling_rate,
            'n_samples': len(eeg_data)
        }
        
        # 1. Band Powers
        print("\n1. Computing band powers...")
        band_powers, relative_powers = self.compute_band_powers(eeg_data)
        results['band_powers'] = band_powers
        results['relative_powers'] = relative_powers
        
        # Print summary
        for band in self.frequency_bands:
            avg_power = np.mean([band_powers[ch][band] for ch in band_powers])
            avg_relative = np.mean([relative_powers[ch][band] for ch in relative_powers])
            print(f"   {band.capitalize():8s}: {avg_power:8.2f} µV² ({avg_relative*100:5.1f}%)")
        
        # 2. Individual Alpha Frequency
        print("\n2. Computing alpha peak frequency (IAF = max Welch PSD in 8–13 Hz)...")
        iaf = self.compute_individual_alpha_frequency(eeg_data)
        results['iaf'] = iaf
        mn, mx = iaf.get("min_across_channels_hz", np.nan), iaf.get("max_across_channels_hz", np.nan)
        if not (np.isnan(mn) or np.isnan(mx)):
            print(
                f"   Mean of 4 channel peaks (TP9,AF7,AF8,TP10): {iaf['overall']:.2f} Hz "
                f"(min {mn:.2f} Hz, max {mx:.2f} Hz across those channels)"
            )
        else:
            print(f"   Mean of 4 channel peaks: {iaf['overall']:.2f} Hz")
        fm = iaf.get('frontal_mean_hz', np.nan)
        if not np.isnan(fm):
            print(f"   Frontal mean (AF7+AF8 only, reference): {fm:.2f} Hz")
        print("   Per channel:", end=" ")
        print(
            ", ".join(
                f"{ch}={iaf['per_channel'][ch]:.2f}"
                for ch in self.eeg_channels
                if ch in iaf["per_channel"] and not np.isnan(iaf["per_channel"][ch])
            )
        )
        
        # 3. Frontal Alpha Asymmetry
        print("\n3. Computing Frontal Alpha Asymmetry (FAA)...")
        faa = self.compute_frontal_alpha_asymmetry(eeg_data)
        results['faa'] = faa
        print(f"   FAA: {faa['faa']:.3f} ({faa['interpretation']})")
        
        # 4. Frequency Ratios
        print("\n4. Computing frequency ratios...")
        ratios = self.compute_frequency_ratios(eeg_data)
        results['ratios'] = ratios
        print(f"   Theta/Beta (attention): {ratios['theta_beta']['average']:.3f}")
        print(f"   Alpha/Beta (relaxation): {ratios['alpha_beta']['average']:.3f}")
        print(f"   Theta/Alpha (drowsiness): {ratios['theta_alpha']['average']:.3f}")
        
        # 5. Aperiodic Slope
        print("\n5. Computing aperiodic (1/f) slope...")
        aperiodic = self.compute_aperiodic_slope(eeg_data, 'AF7')
        results['aperiodic'] = aperiodic
        print(f"   1/f Slope: {aperiodic['slope']:.3f}")
        
        # 6. Complexity Measures
        print("\n6. Computing complexity measures...")
        complexity = self.compute_complexity_measures(eeg_data)
        results['complexity'] = complexity
        print(f"   Sample Entropy (avg): {complexity['average']['sample_entropy']:.3f}")
        print(f"   Higuchi FD (avg): {complexity['average']['higuchi_fd']:.3f}")
        
        print(f"\n{'='*60}")
        
        return results
    
    def compare_pre_post(self, pre_results, post_results, baseline_variability=None):
        """
        Compare pre and post experiment resting state metrics.
        
        Args:
            pre_results: Results from baseline/pre analysis
            post_results: Results from post-distraction/post analysis  
            baseline_variability: Optional dict with SDs from baseline period.
                                  If provided, fatigue index uses personalized normalization.
        
        Returns:
            Dictionary with change metrics and interpretations
        """
        comparison = {
            'pre': pre_results,
            'post': post_results,
            'changes': {},
            'interpretations': []
        }
        
        print(f"\n{'='*70}")
        print("PRE vs POST EXPERIMENT COMPARISON")
        print(f"{'='*70}")
        
        # 1. Band Power Changes
        print("\n1. BAND POWER CHANGES:")
        print("-" * 50)
        
        for band in self.frequency_bands:
            pre_avg = np.mean([pre_results['band_powers'][ch][band] 
                              for ch in pre_results['band_powers']])
            post_avg = np.mean([post_results['band_powers'][ch][band] 
                               for ch in post_results['band_powers']])
            
            change_pct = ((post_avg - pre_avg) / pre_avg) * 100 if pre_avg > 0 else 0
            comparison['changes'][f'{band}_power'] = change_pct
            
            direction = "↑" if change_pct > 0 else "↓"
            print(f"   {band.capitalize():8s}: {pre_avg:8.2f} → {post_avg:8.2f} ({direction}{abs(change_pct):5.1f}%)")
        
        # Interpret band changes
        theta_change = comparison['changes'].get('theta_power', 0)
        alpha_change = comparison['changes'].get('alpha_power', 0)
        
        if theta_change > 10 and alpha_change < -5:
            comparison['interpretations'].append(
                "⚠️ MENTAL FATIGUE: Increased theta + decreased alpha indicates cognitive fatigue"
            )
        elif alpha_change > 10:
            comparison['interpretations'].append(
                "✓ RELAXATION: Increased alpha suggests successful relaxation/recovery"
            )
        
        # 2. Alpha peak frequency (IAF): PSD maximum in 8–13 Hz per channel; summary = grand mean and frontal mean
        print("\n2. ALPHA PEAK FREQUENCY (IAF, 8–13 Hz PSD PEAK):")
        print("-" * 50)
        
        pre_iaf = pre_results['iaf']['overall']
        post_iaf = post_results['iaf']['overall']
        iaf_change = post_iaf - pre_iaf
        comparison['changes']['iaf'] = iaf_change

        pre_iaf_f = pre_results['iaf'].get('frontal_mean_hz', np.nan)
        post_iaf_f = post_results['iaf'].get('frontal_mean_hz', np.nan)
        iaf_frontal_change = post_iaf_f - pre_iaf_f
        comparison['changes']['iaf_frontal'] = iaf_frontal_change
        
        direction = "↑" if iaf_change > 0 else "↓"
        print(
            f"   Mean of 4 channel peaks: {pre_iaf:.2f} Hz → {post_iaf:.2f} Hz "
            f"({direction}{abs(iaf_change):.2f} Hz)"
        )
        if not (np.isnan(pre_iaf_f) or np.isnan(post_iaf_f)):
            dirf = "↑" if iaf_frontal_change > 0 else "↓"
            print(
                f"   Frontal mean (AF7+AF8 peaks): {pre_iaf_f:.2f} Hz → {post_iaf_f:.2f} Hz "
                f"({dirf}{abs(iaf_frontal_change):.2f} Hz)"
            )
        
        if iaf_change < -0.5:
            comparison['interpretations'].append(
                "⚠️ IAF SLOWING: Decreased alpha frequency suggests cognitive fatigue"
            )
        elif iaf_change > 0.5:
            comparison['interpretations'].append(
                "✓ IAF INCREASE: Faster alpha frequency may indicate heightened alertness"
            )
        
        # 3. Frontal Alpha Asymmetry
        print("\n3. FRONTAL ALPHA ASYMMETRY:")
        print("-" * 50)
        
        pre_faa = pre_results['faa']['faa']
        post_faa = post_results['faa']['faa']
        faa_change = post_faa - pre_faa
        comparison['changes']['faa'] = faa_change
        
        print(f"   FAA: {pre_faa:.3f} → {post_faa:.3f} (Δ = {faa_change:+.3f})")
        print(f"   Pre:  {pre_results['faa']['interpretation']}")
        print(f"   Post: {post_results['faa']['interpretation']}")
        
        if faa_change < -0.2:
            comparison['interpretations'].append(
                "⚠️ WITHDRAWAL SHIFT: FAA became more negative - possible negative affect/avoidance"
            )
        elif faa_change > 0.2:
            comparison['interpretations'].append(
                "✓ APPROACH SHIFT: FAA became more positive - possible positive affect/engagement"
            )
        
        # 4. Frequency Ratios
        print("\n4. FREQUENCY RATIOS:")
        print("-" * 50)
        
        ratio_names = ['theta_beta', 'alpha_beta', 'theta_alpha']
        ratio_labels = ['Theta/Beta', 'Alpha/Beta', 'Theta/Alpha']
        
        for rname, rlabel in zip(ratio_names, ratio_labels):
            pre_val = pre_results['ratios'][rname]['average']
            post_val = post_results['ratios'][rname]['average']
            change_pct = ((post_val - pre_val) / pre_val) * 100 if pre_val > 0 else 0
            comparison['changes'][rname] = change_pct
            
            direction = "↑" if change_pct > 0 else "↓"
            print(f"   {rlabel:12s}: {pre_val:.3f} → {post_val:.3f} ({direction}{abs(change_pct):5.1f}%)")
        
        # Interpret ratio changes
        tb_change = comparison['changes'].get('theta_beta', 0)
        if tb_change > 15:
            comparison['interpretations'].append(
                "⚠️ ATTENTION DECLINE: Theta/Beta ratio increased significantly"
            )
        elif tb_change < -15:
            comparison['interpretations'].append(
                "✓ ATTENTION IMPROVEMENT: Theta/Beta ratio decreased"
            )
        
        # 5. Aperiodic Slope
        print("\n5. APERIODIC (1/f) SLOPE:")
        print("-" * 50)
        
        pre_slope = pre_results['aperiodic']['slope']
        post_slope = post_results['aperiodic']['slope']
        slope_change = post_slope - pre_slope
        comparison['changes']['aperiodic_slope'] = slope_change
        
        print(f"   Slope: {pre_slope:.3f} → {post_slope:.3f} (Δ = {slope_change:+.3f})")
        
        if slope_change > 0.2:
            comparison['interpretations'].append(
                "⚠️ FLATTER SLOPE: Increased neural noise, possibly reduced E/I balance"
            )
        
        # 6. Complexity
        print("\n6. COMPLEXITY MEASURES:")
        print("-" * 50)
        
        pre_se = pre_results['complexity']['average']['sample_entropy']
        post_se = post_results['complexity']['average']['sample_entropy']
        se_change = ((post_se - pre_se) / pre_se) * 100 if pre_se > 0 else 0
        comparison['changes']['sample_entropy'] = se_change
        
        pre_hfd = pre_results['complexity']['average']['higuchi_fd']
        post_hfd = post_results['complexity']['average']['higuchi_fd']
        hfd_change = ((post_hfd - pre_hfd) / pre_hfd) * 100 if pre_hfd > 0 else 0
        comparison['changes']['higuchi_fd'] = hfd_change
        
        direction_se = "↑" if se_change > 0 else "↓"
        direction_hfd = "↑" if hfd_change > 0 else "↓"
        print(f"   Sample Entropy: {pre_se:.3f} → {post_se:.3f} ({direction_se}{abs(se_change):5.1f}%)")
        print(f"   Higuchi FD:     {pre_hfd:.3f} → {post_hfd:.3f} ({direction_hfd}{abs(hfd_change):5.1f}%)")
        
        if se_change < -10:
            comparison['interpretations'].append(
                "⚠️ REDUCED COMPLEXITY: Decreased entropy may indicate fatigue"
            )
        
        # 7. Mental Fatigue Index (with Baseline Variability Normalization)
        print("\n" + "="*70)
        print("MENTAL FATIGUE INDEX")
        print("="*70)
        
        # Determine normalization approach
        if baseline_variability is not None:
            print("\n   Using PERSONALIZED normalization (based on participant's baseline variability)")
            use_personalized = True
        else:
            print("\n   Using HEURISTIC normalization (fixed thresholds)")
            use_personalized = False
        
        # Compute raw changes (absolute values for relevant metrics)
        pre_tb = pre_results['ratios']['theta_beta']['average']
        post_tb = post_results['ratios']['theta_beta']['average']
        tb_abs_change = post_tb - pre_tb  # Positive = increase = fatigue
        
        pre_iaf_val = pre_results['iaf']['overall']
        post_iaf_val = post_results['iaf']['overall']
        iaf_abs_change = post_iaf_val - pre_iaf_val  # Negative = decrease = fatigue
        
        # Alpha relative power (average across channels)
        pre_alpha_vals = [pre_results['relative_powers'][ch]['alpha'] * 100 
                         for ch in pre_results['relative_powers'] 
                         if isinstance(pre_results['relative_powers'][ch], dict)]
        post_alpha_vals = [post_results['relative_powers'][ch]['alpha'] * 100 
                          for ch in post_results['relative_powers'] 
                          if isinstance(post_results['relative_powers'][ch], dict)]
        pre_alpha_rel = np.mean(pre_alpha_vals) if pre_alpha_vals else 0
        post_alpha_rel = np.mean(post_alpha_vals) if post_alpha_vals else 0
        alpha_abs_change = post_alpha_rel - pre_alpha_rel  # Negative = decrease = fatigue
        
        pre_se_val = pre_results['complexity']['average']['sample_entropy']
        post_se_val = post_results['complexity']['average']['sample_entropy']
        se_abs_change = post_se_val - pre_se_val  # Negative = decrease = fatigue
        
        # Compute normalized fatigue scores for each indicator
        fatigue_indicators = []
        fatigue_details = []
        
        if use_personalized:
            # PERSONALIZED NORMALIZATION: Change expressed in SD units
            # A change of 2+ SD is considered significant
            sd_threshold = 2.0  # Changes > 2 SD are "fully fatigued" for that metric
            
            # Theta/Beta (increase = fatigue)
            tb_sd = baseline_variability.get('theta_beta_sd', 0.10)
            if tb_sd > 0.001:  # Avoid division by tiny numbers
                tb_z = tb_abs_change / tb_sd
                if tb_z > 0:  # Only count if change is in fatigue direction
                    score = min(1.0, tb_z / sd_threshold)
                    fatigue_indicators.append(score)
                    fatigue_details.append(f"θ/β: +{tb_abs_change:.3f} = {tb_z:.1f} SD → {score*100:.0f}%")
            
            # IAF (decrease = fatigue)
            iaf_sd = baseline_variability.get('iaf_sd', 0.30)
            if iaf_sd > 0.001:
                iaf_z = -iaf_abs_change / iaf_sd  # Negative change means positive z
                if iaf_z > 0:
                    score = min(1.0, iaf_z / sd_threshold)
                    fatigue_indicators.append(score)
                    fatigue_details.append(f"IAF: {iaf_abs_change:.2f} Hz = {iaf_z:.1f} SD → {score*100:.0f}%")
            
            # Alpha Power (decrease = fatigue)
            alpha_sd = baseline_variability.get('alpha_relative_sd', 5.0)
            if alpha_sd > 0.001:
                alpha_z = -alpha_abs_change / alpha_sd
                if alpha_z > 0:
                    score = min(1.0, alpha_z / sd_threshold)
                    fatigue_indicators.append(score)
                    fatigue_details.append(f"Alpha: {alpha_abs_change:.1f}% = {alpha_z:.1f} SD → {score*100:.0f}%")
            
            # Sample Entropy (decrease = fatigue)
            se_sd = baseline_variability.get('sample_entropy_sd', 0.05)
            if se_sd > 0.001:
                se_z = -se_abs_change / se_sd
                if se_z > 0:
                    score = min(1.0, se_z / sd_threshold)
                    fatigue_indicators.append(score)
                    fatigue_details.append(f"Entropy: {se_abs_change:.3f} = {se_z:.1f} SD → {score*100:.0f}%")
            
            print(f"\n   Baseline SDs: θ/β={tb_sd:.3f}, IAF={iaf_sd:.2f}Hz, Alpha={alpha_sd:.1f}%, Entropy={se_sd:.3f}")
            
        else:
            # HEURISTIC NORMALIZATION: Fixed thresholds based on typical values
            
            # Theta/beta increase indicates fatigue (30% change = max)
            if tb_change > 0:
                score = min(1.0, tb_change / 30)
                fatigue_indicators.append(score)
                fatigue_details.append(f"θ/β: +{tb_change:.1f}% → {score*100:.0f}%")
            
            # IAF decrease indicates fatigue (1 Hz change = max)
            if iaf_change < 0:
                score = min(1.0, abs(iaf_change) / 1.0)
                fatigue_indicators.append(score)
                fatigue_details.append(f"IAF: {iaf_change:.2f}Hz → {score*100:.0f}%")
            
            # Alpha decrease indicates fatigue (20% relative change = max)
            if alpha_change < 0:
                score = min(1.0, abs(alpha_change) / 20)
                fatigue_indicators.append(score)
                fatigue_details.append(f"Alpha: {alpha_change:.1f}% → {score*100:.0f}%")
            
            # Entropy decrease indicates fatigue (15% change = max)
            if se_change < 0:
                score = min(1.0, abs(se_change) / 15)
                fatigue_indicators.append(score)
                fatigue_details.append(f"Entropy: {se_change:.1f}% → {score*100:.0f}%")
        
        # Print component details
        if fatigue_details:
            print("\n   Component Contributions:")
            for detail in fatigue_details:
                print(f"      {detail}")
        
        # Calculate final index
        if fatigue_indicators:
            fatigue_index = float(np.mean(fatigue_indicators) * 100)
        else:
            fatigue_index = 0.0
            print("\n   No fatigue indicators detected (all changes in non-fatigue direction)")
        
        comparison['fatigue_index'] = fatigue_index
        comparison['fatigue_details'] = fatigue_details
        comparison['normalization_type'] = 'personalized' if use_personalized else 'heuristic'
        
        print(f"\n   MENTAL FATIGUE INDEX: {fatigue_index:.1f}/100")
        
        if fatigue_index >= 60:
            print("   Interpretation: SIGNIFICANT FATIGUE - participant showed substantial cognitive depletion")
        elif fatigue_index >= 40:
            print("   Interpretation: MODERATE FATIGUE - some cognitive resource depletion evident")
        elif fatigue_index >= 20:
            print("   Interpretation: MILD FATIGUE - minor effects observed")
        else:
            print("   Interpretation: MINIMAL FATIGUE - participant maintained cognitive resources well")
        
        # 8. Resting state change index (bidirectional magnitude, 0–100)
        print("\n" + "="*70)
        print("RESTING STATE CHANGE INDEX (BIDIRECTIONAL)")
        print("="*70)
        
        change_indicators = []
        change_index_details = []
        
        if use_personalized:
            sd_threshold_ci = 2.0
            
            tb_sd_ci = baseline_variability.get('theta_beta_sd', 0.10)
            if tb_sd_ci > 0.001:
                z_mag_tb = abs(tb_abs_change) / tb_sd_ci
                score_ci = min(1.0, z_mag_tb / sd_threshold_ci)
                change_indicators.append(score_ci)
                change_index_details.append(
                    f"θ/β: Δ={tb_abs_change:+.3f} = |z|={z_mag_tb:.1f} SD → {score_ci*100:.0f}%"
                )
            
            iaf_sd_ci = baseline_variability.get('iaf_sd', 0.30)
            if iaf_sd_ci > 0.001:
                z_mag_iaf = abs(iaf_abs_change) / iaf_sd_ci
                score_ci = min(1.0, z_mag_iaf / sd_threshold_ci)
                change_indicators.append(score_ci)
                change_index_details.append(
                    f"IAF: Δ={iaf_abs_change:+.2f} Hz = |z|={z_mag_iaf:.1f} SD → {score_ci*100:.0f}%"
                )
            
            alpha_sd_ci = baseline_variability.get('alpha_relative_sd', 5.0)
            if alpha_sd_ci > 0.001:
                z_mag_al = abs(alpha_abs_change) / alpha_sd_ci
                score_ci = min(1.0, z_mag_al / sd_threshold_ci)
                change_indicators.append(score_ci)
                change_index_details.append(
                    f"Alpha: Δ={alpha_abs_change:+.1f}% = |z|={z_mag_al:.1f} SD → {score_ci*100:.0f}%"
                )
            
            se_sd_ci = baseline_variability.get('sample_entropy_sd', 0.05)
            if se_sd_ci > 0.001:
                z_mag_se = abs(se_abs_change) / se_sd_ci
                score_ci = min(1.0, z_mag_se / sd_threshold_ci)
                change_indicators.append(score_ci)
                change_index_details.append(
                    f"Entropy: Δ={se_abs_change:+.3f} = |z|={z_mag_se:.1f} SD → {score_ci*100:.0f}%"
                )
        else:
            score_ci = min(1.0, abs(tb_change) / 30.0)
            change_indicators.append(score_ci)
            change_index_details.append(f"θ/β: |{tb_change:.1f}|% → {score_ci*100:.0f}%")
            
            score_ci = min(1.0, abs(iaf_change) / 1.0)
            change_indicators.append(score_ci)
            change_index_details.append(f"IAF: |{iaf_change:.2f}| Hz → {score_ci*100:.0f}%")
            
            score_ci = min(1.0, abs(alpha_change) / 20.0)
            change_indicators.append(score_ci)
            change_index_details.append(f"Alpha: |{alpha_change:.1f}|% → {score_ci*100:.0f}%")
            
            score_ci = min(1.0, abs(se_change) / 15.0)
            change_indicators.append(score_ci)
            change_index_details.append(f"Entropy: |{se_change:.1f}|% → {score_ci*100:.0f}%")
        
        if change_indicators:
            resting_state_change_index = float(np.mean(change_indicators) * 100)
        else:
            resting_state_change_index = 0.0
        
        comparison['resting_state_change_index'] = resting_state_change_index
        comparison['change_index_details'] = change_index_details
        
        print(f"\n   RESTING STATE CHANGE INDEX: {resting_state_change_index:.1f}/100")
        if change_index_details:
            print("\n   Change index component contributions:")
            for detail in change_index_details:
                print(f"      {detail}")
        
        # Print all interpretations
        print("\n" + "="*70)
        print("SUMMARY OF FINDINGS")
        print("="*70)
        
        for interp in comparison['interpretations']:
            print(f"  {interp}")
        
        if not comparison['interpretations']:
            print("  No significant changes detected between pre and post recordings.")
        
        return comparison
    
    def visualize_comparison(self, pre_results, post_results, comparison, save_path=None):
        """
        Create visualization comparing pre and post resting state metrics.
        """
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        fig.suptitle('Pre vs Post Experiment Resting State EEG Comparison', fontsize=14, fontweight='bold')
        
        # Helper for safe float conversion
        def _safe_float(value, default=0.0):
            try:
                result = float(value)
                if np.isnan(result) or np.isinf(result):
                    return default
                return result
            except (TypeError, ValueError):
                return default
        
        # 1. Band Powers Comparison
        ax1 = axes[0, 0]
        bands = list(self.frequency_bands.keys())
        pre_powers = [_safe_float(np.mean([pre_results['band_powers'][ch][b] for ch in pre_results['band_powers']])) 
                      for b in bands]
        post_powers = [_safe_float(np.mean([post_results['band_powers'][ch][b] for ch in post_results['band_powers']])) 
                       for b in bands]
        
        x = np.arange(len(bands))
        width = 0.35
        ax1.bar(x - width/2, pre_powers, width, label='Pre', color='steelblue', alpha=0.8)
        ax1.bar(x + width/2, post_powers, width, label='Post', color='coral', alpha=0.8)
        ax1.set_xlabel('Frequency Band')
        ax1.set_ylabel('Power (µV²)')
        ax1.set_title('Band Powers')
        ax1.set_xticks(x)
        ax1.set_xticklabels([b.capitalize() for b in bands])
        ax1.legend()
        
        # 2. Relative Powers (Pie charts)
        ax2 = axes[0, 1]
        pre_relative = [_safe_float(np.mean([pre_results['relative_powers'][ch][b] for ch in pre_results['relative_powers']]), 0.01) 
                        for b in bands]
        colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A0572']
        ax2.pie(pre_relative, labels=[b.capitalize() for b in bands], autopct='%1.1f%%', colors=colors, startangle=90)
        ax2.set_title('Pre: Relative Band Powers')
        
        ax3 = axes[0, 2]
        post_relative = [_safe_float(np.mean([post_results['relative_powers'][ch][b] for ch in post_results['relative_powers']]), 0.01) 
                         for b in bands]
        ax3.pie(post_relative, labels=[b.capitalize() for b in bands], autopct='%1.1f%%', colors=colors, startangle=90)
        ax3.set_title('Post: Relative Band Powers')
        
        # 3. Frequency Ratios
        ax4 = axes[1, 0]
        ratio_names = ['theta_beta', 'alpha_beta', 'theta_alpha']
        ratio_labels = ['θ/β\n(Attention)', 'α/β\n(Relaxation)', 'θ/α\n(Drowsiness)']
        pre_ratios = [_safe_float(pre_results['ratios'][r]['average']) for r in ratio_names]
        post_ratios = [_safe_float(post_results['ratios'][r]['average']) for r in ratio_names]
        
        x = np.arange(len(ratio_names))
        ax4.bar(x - width/2, pre_ratios, width, label='Pre', color='steelblue', alpha=0.8)
        ax4.bar(x + width/2, post_ratios, width, label='Post', color='coral', alpha=0.8)
        ax4.set_xlabel('Ratio')
        ax4.set_ylabel('Value')
        ax4.set_title('Frequency Ratios')
        ax4.set_xticks(x)
        ax4.set_xticklabels(ratio_labels)
        ax4.legend()
        
        # 4. Key Metrics Comparison
        ax5 = axes[1, 1]
        metrics = ['IAF', 'FAA', 'Entropy', 'Fatigue']
        pre_vals = [
            _safe_float(pre_results['iaf']['overall']),
            _safe_float(pre_results['faa']['faa']) * 10,  # Scale for visibility
            _safe_float(pre_results['complexity']['average']['sample_entropy']) * 10,
            0.0  # No fatigue before experiment
        ]
        post_vals = [
            _safe_float(post_results['iaf']['overall']),
            _safe_float(post_results['faa']['faa']) * 10,
            _safe_float(post_results['complexity']['average']['sample_entropy']) * 10,
            _safe_float(comparison['fatigue_index']) / 10
        ]
        
        x = np.arange(len(metrics))
        ax5.bar(x - width/2, pre_vals, width, label='Pre', color='steelblue', alpha=0.8)
        ax5.bar(x + width/2, post_vals, width, label='Post', color='coral', alpha=0.8)
        ax5.set_xlabel('Metric')
        ax5.set_ylabel('Value (scaled)')
        ax5.set_title('Key Metrics')
        ax5.set_xticks(x)
        ax5.set_xticklabels(metrics)
        ax5.legend()
        
        # 5. Fatigue Index Gauge
        ax6 = axes[1, 2]
        fatigue = _safe_float(comparison['fatigue_index'])
        
        # Create a simple gauge
        theta = np.linspace(0, np.pi, 100)
        r = 1
        ax6.fill_between(theta * 180 / np.pi, 0, 1, alpha=0.1)
        
        # Colored sections
        colors_gauge = ['#2ecc71', '#f1c40f', '#e74c3c']
        sections = [(0, 60), (60, 120), (120, 180)]
        for (start, end), color in zip(sections, colors_gauge):
            section_theta = np.linspace(start * np.pi/180, end * np.pi/180, 50)
            ax6.fill_between(section_theta * 180/np.pi, 0.3, 1, alpha=0.3, color=color)
        
        # Needle
        needle_angle = (180 - fatigue * 1.8) * np.pi / 180
        ax6.annotate('', xy=(needle_angle * 180/np.pi, 0.9), xytext=(90, 0),
                    arrowprops=dict(arrowstyle='->', color='black', lw=3))
        
        ax6.set_xlim(0, 180)
        ax6.set_ylim(0, 1.2)
        ax6.set_title(f'Mental Fatigue Index: {fatigue:.1f}/100')
        ax6.axis('off')
        ax6.text(90, -0.1, f'{fatigue:.1f}', ha='center', va='center', fontsize=24, fontweight='bold')
        ax6.text(10, 0.5, 'Low', fontsize=10, color='green')
        ax6.text(170, 0.5, 'High', fontsize=10, color='red')
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
            print(f"\nVisualization saved to: {save_path}")
            plt.close()
        else:
            plt.show()
    
    def generate_report(self, pre_results, post_results, comparison, subject_id, save_path=None):
        """
        Generate a comprehensive text report of the resting state analysis.
        """
        report = []
        report.append("="*80)
        report.append("RESTING STATE EEG ANALYSIS REPORT")
        report.append("="*80)
        report.append(f"Subject ID: {subject_id}")
        report.append(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report.append(f"Pre-experiment recording: {pre_results['duration_seconds']:.1f} seconds")
        report.append(f"Post-experiment recording: {post_results['duration_seconds']:.1f} seconds")
        report.append("")
        
        report.append("-"*80)
        report.append("1. BAND POWER ANALYSIS")
        report.append("-"*80)
        report.append("")
        report.append(f"{'Band':<10} {'Pre (µV²)':<15} {'Post (µV²)':<15} {'Change':<15}")
        report.append("-"*55)
        
        for band in self.frequency_bands:
            pre_avg = np.mean([pre_results['band_powers'][ch][band] for ch in pre_results['band_powers']])
            post_avg = np.mean([post_results['band_powers'][ch][band] for ch in post_results['band_powers']])
            change = comparison['changes'].get(f'{band}_power', 0)
            report.append(f"{band.capitalize():<10} {pre_avg:<15.2f} {post_avg:<15.2f} {change:+.1f}%")
        
        report.append("")
        report.append("-"*80)
        report.append("2. ALPHA PEAK FREQUENCY — 8–13 Hz PSD PEAK (TP9, AF7, AF8, TP10)")
        report.append("-"*80)
        pre_iaf = pre_results['iaf']
        post_iaf = post_results['iaf']
        report.append("Per-channel alpha peak (Hz) = frequency bin with maximum Welch PSD in 8–13 Hz.")
        report.append("")
        report.append(f"{'Channel':<8} {'Pre (Hz)':<12} {'Post (Hz)':<12}")
        report.append("-" * 32)
        all_ch = sorted(
            set(pre_iaf['per_channel'].keys()) | set(post_iaf['per_channel'].keys()),
            key=lambda x: (self.eeg_channels.index(x) if x in self.eeg_channels else 99, x),
        )
        for ch in all_ch:
            pv = pre_iaf['per_channel'].get(ch, np.nan)
            qv = post_iaf['per_channel'].get(ch, np.nan)
            ps = f"{pv:.2f}" if not np.isnan(pv) else "N/A"
            qs = f"{qv:.2f}" if not np.isnan(qv) else "N/A"
            report.append(f"{ch:<8} {ps:<12} {qs:<12}")
        report.append("")
        report.append(
            f"Mean of 4 channel peaks: {pre_iaf['overall']:.2f} Hz → {post_iaf['overall']:.2f} Hz"
        )
        report.append(
            f"  Within-subject range (pre):  min {pre_iaf.get('min_across_channels_hz', np.nan):.2f} Hz, "
            f"max {pre_iaf.get('max_across_channels_hz', np.nan):.2f} Hz across TP9/AF7/AF8/TP10"
        )
        report.append(
            f"  Within-subject range (post): min {post_iaf.get('min_across_channels_hz', np.nan):.2f} Hz, "
            f"max {post_iaf.get('max_across_channels_hz', np.nan):.2f} Hz"
        )
        report.append(
            f"Frontal mean AF7+AF8 (ref): {pre_iaf.get('frontal_mean_hz', np.nan):.2f} Hz → "
            f"{post_iaf.get('frontal_mean_hz', np.nan):.2f} Hz"
        )
        report.append(f"Δ mean of 4 peaks (post − pre): {comparison['changes']['iaf']:+.2f} Hz")
        iaf_fr_d = comparison["changes"].get("iaf_frontal", np.nan)
        if not np.isnan(iaf_fr_d):
            report.append(f"Δ frontal mean AF7+AF8 (post − pre): {iaf_fr_d:+.2f} Hz")
        report.append("")
        
        report.append("-"*80)
        report.append("3. FRONTAL ALPHA ASYMMETRY (FAA)")
        report.append("-"*80)
        report.append(f"Pre-experiment FAA:  {pre_results['faa']['faa']:.3f} ({pre_results['faa']['interpretation']})")
        report.append(f"Post-experiment FAA: {post_results['faa']['faa']:.3f} ({post_results['faa']['interpretation']})")
        report.append(f"Change: {comparison['changes']['faa']:+.3f}")
        report.append("")
        
        report.append("-"*80)
        report.append("4. FREQUENCY RATIOS")
        report.append("-"*80)
        report.append(f"{'Ratio':<15} {'Pre':<10} {'Post':<10} {'Change':<10}")
        report.append("-"*45)
        
        for rname in ['theta_beta', 'alpha_beta', 'theta_alpha']:
            pre_val = pre_results['ratios'][rname]['average']
            post_val = post_results['ratios'][rname]['average']
            change = comparison['changes'].get(rname, 0)
            report.append(f"{rname:<15} {pre_val:<10.3f} {post_val:<10.3f} {change:+.1f}%")
        
        report.append("")
        report.append("-"*80)
        report.append("5. COMPLEXITY MEASURES")
        report.append("-"*80)
        report.append(f"Sample Entropy: {pre_results['complexity']['average']['sample_entropy']:.3f} → {post_results['complexity']['average']['sample_entropy']:.3f}")
        report.append(f"Higuchi FD:     {pre_results['complexity']['average']['higuchi_fd']:.3f} → {post_results['complexity']['average']['higuchi_fd']:.3f}")
        report.append("")
        
        report.append("="*80)
        report.append("MENTAL FATIGUE INDEX")
        report.append("="*80)
        report.append(f"Score: {comparison['fatigue_index']:.1f} / 100")
        
        if comparison['fatigue_index'] >= 60:
            report.append("Interpretation: SIGNIFICANT FATIGUE")
        elif comparison['fatigue_index'] >= 40:
            report.append("Interpretation: MODERATE FATIGUE")
        elif comparison['fatigue_index'] >= 20:
            report.append("Interpretation: MILD FATIGUE")
        else:
            report.append("Interpretation: MINIMAL FATIGUE")
        
        report.append("")
        report.append("="*80)
        report.append("KEY FINDINGS")
        report.append("="*80)
        
        for interp in comparison['interpretations']:
            report.append(f"  {interp}")
        
        if not comparison['interpretations']:
            report.append("  No significant changes detected.")
        
        report.append("")
        report.append("="*80)
        
        report_text = "\n".join(report)
        
        if save_path:
            with open(save_path, 'w') as f:
                f.write(report_text)
            print(f"\nReport saved to: {save_path}")
        
        return report_text


In [ ]:
def find_task_eeg_files(base_path, level_id, subject_id):
    """
    Find EEG files for a specific level and subject across all sets.
    
    Returns:
        Dictionary with set_id -> list of file paths
    """
    files_by_set = {1: [], 2: [], 3: []}
    
    # Try different path patterns
    xdf_paths = [
        os.path.join(base_path, f"Level{level_id}", "Results_XDF", f"Subject{subject_id}"),
        os.path.join(base_path, f"Level{level_id}", "Results_XDF", f"subject{subject_id}"),
        os.path.join(base_path, "Results_XDF", f"Subject{subject_id}"),
    ]
    
    for xdf_path in xdf_paths:
        if os.path.exists(xdf_path):
            for filename in os.listdir(xdf_path):
                if filename.endswith('.xdf') or filename.endswith('.csv'):
                    filepath = os.path.join(xdf_path, filename)
                    
                    # Determine set from filename
                    for set_id in [1, 2, 3]:
                        if f'set{set_id}' in filename.lower() or f'set{{{set_id}}}' in filename:
                            files_by_set[set_id].append(filepath)
                            break
    
    return files_by_set


In [ ]:
def compare_task_sets(analyzer, set_results, subject_id, save_path=None):
    """
    Compare EEG metrics across task sets (Baseline, Set2, Set3).
    
    Args:
        analyzer: RestingStateAnalyzer instance
        set_results: Dictionary with set_id -> analysis results
        subject_id: Subject identifier
        save_path: Optional path to save visualization
    """
    print(f"\n{'='*70}")
    print("TASK EEG COMPARISON ACROSS SETS")
    print(f"{'='*70}")
    
    available_sets = [s for s in [1, 2, 3] if s in set_results and set_results[s] is not None]
    
    if len(available_sets) < 2:
        print("Need at least 2 sets to compare")
        return None
    
    set_labels = {
        1: 'Baseline (Set 1)',
        2: 'Distraction+Response (Set 2)',
        3: 'Distraction+NoResponse (Set 3)'
    }
    
    comparison = {
        'sets': available_sets,
        'results': set_results,
        'changes': {},
        'interpretations': []
    }
    
    # Use Set 1 as baseline reference
    baseline = set_results.get(1)
    
    # 1. Band Power Changes
    print("\n1. BAND POWER CHANGES (relative to Baseline):")
    print("-" * 70)
    print(f"{'Band':<10} {'Baseline':<12} {'Set 2':<12} {'Set 3':<12} {'Set2 Δ%':<10} {'Set3 Δ%':<10}")
    print("-" * 70)
    
    for band in analyzer.frequency_bands:
        baseline_power = np.mean([baseline['band_powers'][ch][band] 
                                  for ch in baseline['band_powers']]) if baseline else 0
        
        row = [band.capitalize(), f"{baseline_power:.2f}"]
        
        for set_id in [2, 3]:
            if set_id in set_results and set_results[set_id]:
                set_power = np.mean([set_results[set_id]['band_powers'][ch][band] 
                                    for ch in set_results[set_id]['band_powers']])
                change_pct = ((set_power - baseline_power) / baseline_power * 100) if baseline_power > 0 else 0
                row.append(f"{set_power:.2f}")
                comparison['changes'][f'{band}_set{set_id}'] = change_pct
            else:
                row.append("N/A")
        
        # Add change percentages
        for set_id in [2, 3]:
            key = f'{band}_set{set_id}'
            if key in comparison['changes']:
                change = comparison['changes'][key]
                row.append(f"{change:+.1f}%")
            else:
                row.append("N/A")
        
        print(f"{row[0]:<10} {row[1]:<12} {row[2]:<12} {row[3]:<12} {row[4]:<10} {row[5]:<10}")
    
    # 2. Frequency Ratios
    print("\n2. FREQUENCY RATIOS:")
    print("-" * 70)
    print(f"{'Ratio':<15} {'Baseline':<12} {'Set 2':<12} {'Set 3':<12} {'Set2 Δ%':<10} {'Set3 Δ%':<10}")
    print("-" * 70)
    
    for ratio_name in ['theta_beta', 'alpha_beta', 'theta_alpha']:
        baseline_ratio = baseline['ratios'][ratio_name]['average'] if baseline else 0
        
        row = [ratio_name, f"{baseline_ratio:.3f}"]
        
        for set_id in [2, 3]:
            if set_id in set_results and set_results[set_id]:
                set_ratio = set_results[set_id]['ratios'][ratio_name]['average']
                change_pct = ((set_ratio - baseline_ratio) / baseline_ratio * 100) if baseline_ratio > 0 else 0
                row.append(f"{set_ratio:.3f}")
                comparison['changes'][f'{ratio_name}_set{set_id}'] = change_pct
            else:
                row.append("N/A")
        
        for set_id in [2, 3]:
            key = f'{ratio_name}_set{set_id}'
            if key in comparison['changes']:
                change = comparison['changes'][key]
                row.append(f"{change:+.1f}%")
            else:
                row.append("N/A")
        
        print(f"{row[0]:<15} {row[1]:<12} {row[2]:<12} {row[3]:<12} {row[4]:<10} {row[5]:<10}")
    
    # Interpret theta/beta changes
    tb_set2 = comparison['changes'].get('theta_beta_set2', 0)
    tb_set3 = comparison['changes'].get('theta_beta_set3', 0)
    
    if tb_set2 > 15:
        comparison['interpretations'].append(
            "⚠️ Set 2: Theta/Beta increased significantly - distraction impacted attention"
        )
    elif tb_set2 < -10:
        comparison['interpretations'].append(
            "✓ Set 2: Theta/Beta decreased - possible heightened focus due to distraction challenge"
        )
    
    if tb_set3 > 15:
        comparison['interpretations'].append(
            "⚠️ Set 3: Theta/Beta increased - accumulated fatigue or distraction effect"
        )
    
    # 3. Individual Alpha Frequency
    print("\n3. INDIVIDUAL ALPHA FREQUENCY (IAF):")
    print("-" * 70)
    
    baseline_iaf = baseline['iaf']['overall'] if baseline else 0
    print(f"   Baseline: {baseline_iaf:.2f} Hz")
    
    for set_id in [2, 3]:
        if set_id in set_results and set_results[set_id]:
            set_iaf = set_results[set_id]['iaf']['overall']
            change = set_iaf - baseline_iaf
            comparison['changes'][f'iaf_set{set_id}'] = change
            direction = "↑" if change > 0 else "↓"
            print(f"   Set {set_id}: {set_iaf:.2f} Hz ({direction}{abs(change):.2f} Hz from baseline)")
            
            if change < -0.5:
                comparison['interpretations'].append(
                    f"⚠️ Set {set_id}: IAF slowing detected - indicates cognitive fatigue"
                )
    
    # 4. Frontal Alpha Asymmetry
    print("\n4. FRONTAL ALPHA ASYMMETRY (FAA):")
    print("-" * 70)
    
    baseline_faa = baseline['faa']['faa'] if baseline else 0
    print(f"   Baseline: {baseline_faa:.3f} ({baseline['faa']['interpretation'] if baseline else 'N/A'})")
    
    for set_id in [2, 3]:
        if set_id in set_results and set_results[set_id]:
            set_faa = set_results[set_id]['faa']['faa']
            change = set_faa - baseline_faa
            comparison['changes'][f'faa_set{set_id}'] = change
            print(f"   Set {set_id}: {set_faa:.3f} ({set_results[set_id]['faa']['interpretation']}) Δ={change:+.3f}")
    
    # 5. Complexity
    print("\n5. COMPLEXITY MEASURES:")
    print("-" * 70)
    
    baseline_se = baseline['complexity']['average']['sample_entropy'] if baseline else 0
    baseline_hfd = baseline['complexity']['average']['higuchi_fd'] if baseline else 0
    print(f"   Baseline - Sample Entropy: {baseline_se:.3f}, Higuchi FD: {baseline_hfd:.3f}")
    
    for set_id in [2, 3]:
        if set_id in set_results and set_results[set_id]:
            set_se = set_results[set_id]['complexity']['average']['sample_entropy']
            set_hfd = set_results[set_id]['complexity']['average']['higuchi_fd']
            se_change = ((set_se - baseline_se) / baseline_se * 100) if baseline_se > 0 else 0
            hfd_change = ((set_hfd - baseline_hfd) / baseline_hfd * 100) if baseline_hfd > 0 else 0
            print(f"   Set {set_id} - Sample Entropy: {set_se:.3f} ({se_change:+.1f}%), Higuchi FD: {set_hfd:.3f} ({hfd_change:+.1f}%)")
            
            comparison['changes'][f'entropy_set{set_id}'] = se_change
            
            if se_change < -10:
                comparison['interpretations'].append(
                    f"⚠️ Set {set_id}: Reduced signal complexity - possible cognitive fatigue"
                )
    
    # 6. Mental Load / Fatigue Index per Set
    print("\n" + "="*70)
    print("COGNITIVE LOAD INDEX (relative to Baseline)")
    print("="*70)
    
    for set_id in [2, 3]:
        if set_id not in set_results or set_results[set_id] is None:
            continue
            
        load_indicators = []
        
        # Higher theta/beta = more load
        tb_change = comparison['changes'].get(f'theta_beta_set{set_id}', 0)
        if tb_change > 0:
            load_indicators.append(min(1.0, tb_change / 30))
        
        # Lower alpha = more load
        alpha_change = comparison['changes'].get(f'alpha_set{set_id}', 0)
        if alpha_change < 0:
            load_indicators.append(min(1.0, abs(alpha_change) / 20))
        
        # IAF slowing = more load
        iaf_change = comparison['changes'].get(f'iaf_set{set_id}', 0)
        if iaf_change < 0:
            load_indicators.append(min(1.0, abs(iaf_change) / 1.0))
        
        # Reduced complexity = more load
        entropy_change = comparison['changes'].get(f'entropy_set{set_id}', 0)
        if entropy_change < 0:
            load_indicators.append(min(1.0, abs(entropy_change) / 15))
        
        if load_indicators:
            load_index = np.mean(load_indicators) * 100
        else:
            load_index = 0
        
        comparison[f'load_index_set{set_id}'] = load_index
        
        print(f"\n   {set_labels[set_id]}")
        print(f"   COGNITIVE LOAD INDEX: {load_index:.1f}/100")
        
        if load_index >= 50:
            print("   → HIGH cognitive load - significant attentional demands")
        elif load_index >= 30:
            print("   → MODERATE cognitive load - noticeable attentional effort")
        elif load_index >= 15:
            print("   → MILD cognitive load - some attentional demands")
        else:
            print("   → MINIMAL cognitive load - similar to baseline")
    
    # Summary
    print("\n" + "="*70)
    print("SUMMARY OF FINDINGS")
    print("="*70)
    
    for interp in comparison['interpretations']:
        print(f"  {interp}")
    
    if not comparison['interpretations']:
        print("  No significant changes detected across sets.")
    
    return comparison


In [ ]:
def visualize_task_comparison(analyzer, set_results, comparison, subject_id, save_path=None):
    """
    Create visualization comparing EEG metrics across task sets.
    """
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f'EEG Comparison Across Task Sets - Subject {subject_id}', fontsize=14, fontweight='bold')
    
    available_sets = [s for s in [1, 2, 3] if s in set_results and set_results[s] is not None]
    set_labels = {1: 'Baseline', 2: 'Set 2\n(Distract+Resp)', 3: 'Set 3\n(Distract+NoResp)'}
    colors = {1: 'steelblue', 2: 'coral', 3: 'seagreen'}
    
    # 1. Band Powers Comparison
    ax1 = axes[0, 0]
    bands = list(analyzer.frequency_bands.keys())
    x = np.arange(len(bands))
    width = 0.25
    
    for i, set_id in enumerate(available_sets):
        powers = [np.mean([set_results[set_id]['band_powers'][ch][b] 
                          for ch in set_results[set_id]['band_powers']]) for b in bands]
        ax1.bar(x + (i - 1) * width, powers, width, label=set_labels[set_id], color=colors[set_id], alpha=0.8)
    
    ax1.set_xlabel('Frequency Band')
    ax1.set_ylabel('Power (µV²)')
    ax1.set_title('Band Powers')
    ax1.set_xticks(x)
    ax1.set_xticklabels([b.capitalize() for b in bands])
    ax1.legend()
    
    # 2. Frequency Ratios
    ax2 = axes[0, 1]
    ratio_names = ['theta_beta', 'alpha_beta', 'theta_alpha']
    ratio_labels = ['θ/β', 'α/β', 'θ/α']
    x = np.arange(len(ratio_names))
    
    for i, set_id in enumerate(available_sets):
        ratios = [set_results[set_id]['ratios'][r]['average'] for r in ratio_names]
        ax2.bar(x + (i - 1) * width, ratios, width, label=set_labels[set_id], color=colors[set_id], alpha=0.8)
    
    ax2.set_xlabel('Ratio')
    ax2.set_ylabel('Value')
    ax2.set_title('Frequency Ratios')
    ax2.set_xticks(x)
    ax2.set_xticklabels(ratio_labels)
    ax2.legend()
    
    # 3. IAF Comparison
    ax3 = axes[0, 2]
    iafs = [set_results[s]['iaf']['overall'] for s in available_sets]
    bar_colors = [colors[s] for s in available_sets]
    ax3.bar([set_labels[s] for s in available_sets], iafs, color=bar_colors, alpha=0.8)
    ax3.set_ylabel('Frequency (Hz)')
    ax3.set_title('Individual Alpha Frequency')
    ax3.set_ylim(8, 12)
    
    # 4. FAA Comparison
    ax4 = axes[1, 0]
    faas = [set_results[s]['faa']['faa'] for s in available_sets]
    bar_colors = [colors[s] for s in available_sets]
    ax4.bar([set_labels[s] for s in available_sets], faas, color=bar_colors, alpha=0.8)
    ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax4.set_ylabel('FAA Value')
    ax4.set_title('Frontal Alpha Asymmetry')
    
    # 5. Complexity
    ax5 = axes[1, 1]
    metrics = ['Sample Entropy', 'Higuchi FD']
    x = np.arange(len(metrics))
    
    for i, set_id in enumerate(available_sets):
        values = [
            set_results[set_id]['complexity']['average']['sample_entropy'],
            set_results[set_id]['complexity']['average']['higuchi_fd']
        ]
        ax5.bar(x + (i - 1) * width, values, width, label=set_labels[set_id], color=colors[set_id], alpha=0.8)
    
    ax5.set_xlabel('Metric')
    ax5.set_ylabel('Value')
    ax5.set_title('Complexity Measures')
    ax5.set_xticks(x)
    ax5.set_xticklabels(metrics)
    ax5.legend()
    
    # 6. Cognitive Load Index
    ax6 = axes[1, 2]
    
    load_values = []
    load_labels = []
    load_colors = []
    
    for set_id in [2, 3]:
        key = f'load_index_set{set_id}'
        if key in comparison:
            load_values.append(comparison[key])
            load_labels.append(set_labels[set_id])
            load_colors.append(colors[set_id])
    
    if load_values:
        bars = ax6.bar(load_labels, load_values, color=load_colors, alpha=0.8)
        ax6.set_ylabel('Load Index')
        ax6.set_title('Cognitive Load Index\n(relative to baseline)')
        ax6.set_ylim(0, 100)
        
        # Add value labels on bars
        for bar, val in zip(bars, load_values):
            ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                    f'{val:.1f}', ha='center', va='bottom', fontsize=12)
        
        # Add reference lines
        ax6.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='High load threshold')
        ax6.axhline(y=30, color='orange', linestyle='--', alpha=0.5, label='Moderate load threshold')
    else:
        ax6.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax6.transAxes)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"\nVisualization saved to: {save_path}")
        plt.close()
    else:
        plt.show()


In [ ]:
def safe_float(value, default=0.0):
    """Safely convert a value to Python float, handling numpy types and NaN."""
    try:
        result = float(value)
        if np.isnan(result) or np.isinf(result):
            return default
        return result
    except (TypeError, ValueError):
        return default


In [ ]:
def main():
    """Main function for interactive EEG state analysis."""
    print("="*70)
    print("EEG STATE ANALYSIS")
    print("Resting State & Task EEG Comparison")
    print("="*70)
    print("\nAnalysis Modes:")
    print("  1. Task EEG Analysis (Compare Sets 1, 2, 3 from a level)")
    print("  2. Resting State Analysis (Pre/Post experiment)")
    print("  3. Combined Analysis (Resting + Task EEG)")
    print()
    
    while True:
        mode = input("Select mode (1, 2, or 3): ").strip()
        if mode in ['1', '2', '3']:
            break
        print("Please enter 1, 2, or 3.")
    
    analyzer = RestingStateAnalyzer()
    base_path = str(_ANALYSIS_DIR)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # ========================================
    # MODE 1 or 3: Task EEG Analysis
    # ========================================
    if mode in ['1', '3']:
        print("\n" + "="*70)
        print("TASK EEG ANALYSIS")
        print("Compare baseline vs distraction sets within a task")
        print("="*70)
        
        # Get subject ID
        subject_id = input("\nEnter Subject ID: ").strip()
        if not subject_id:
            subject_id = "Unknown"
        
        # Task names by level
        task_names_by_level = {
            1: {1: 'Classic Go/No-Go', 2: 'SART', 3: 'Semantic'},
            2: {1: 'Selective Visual', 2: 'Color-Number', 3: 'Spatial Cue'},
            3: {1: 'Picture Recall', 2: 'Color-Location', 3: 'Dual Task'}
        }
        
        # Get level
        print("\nSelect Level:")
        print("  1. Level 1 (Classic, SART, Semantic)")
        print("  2. Level 2 (Selective Visual, Color-Number, Spatial Cue)")
        print("  3. Level 3 (Picture Recall, Color-Location, Dual Task)")
        
        while True:
            level_input = input("Enter level (1, 2, or 3): ").strip()
            if level_input in ['1', '2', '3']:
                level_id = int(level_input)
                break
            print("Please enter 1, 2, or 3.")
        
        # Get task
        print(f"\nSelect Task for Level {level_id}:")
        for task_id, task_name in task_names_by_level[level_id].items():
            print(f"  {task_id}. {task_name}")
        
        while True:
            task_input = input("Enter task (1, 2, or 3): ").strip()
            if task_input in ['1', '2', '3']:
                task_id = int(task_input)
                break
            print("Please enter 1, 2, or 3.")
        
        task_name = task_names_by_level[level_id][task_id]
        
        print(f"\n{'='*70}")
        print(f"TASK EEG ANALYSIS: Level {level_id} - {task_name}")
        print(f"Subject: {subject_id}")
        print(f"{'='*70}")
        
        # Find EEG files for this specific task
        xdf_paths = [
            os.path.join(base_path, f"Level{level_id}", "Results_XDF", f"Subject{subject_id}"),
            os.path.join(base_path, f"Level{level_id}", "Results_XDF", f"subject{subject_id}"),
            os.path.join(base_path, "Results_XDF", f"Subject{subject_id}"),
        ]
        
        set_files = {1: None, 2: None, 3: None}
        
        for xdf_path in xdf_paths:
            if os.path.exists(xdf_path):
                for filename in os.listdir(xdf_path):
                    if filename.endswith('.xdf') or filename.endswith('.csv'):
                        # Check if this file matches our task
                        if f'task{{{task_id}}}' in filename or f'task{task_id}' in filename.lower():
                            filepath = os.path.join(xdf_path, filename)
                            
                            # Determine set
                            for set_id in [1, 2, 3]:
                                if f'set{{{set_id}}}' in filename or f'set{set_id}' in filename.lower():
                                    if set_files[set_id] is None:
                                        set_files[set_id] = filepath
                                    break
        
        # Load and analyze each set
        set_results = {}
        set_eeg_data = {}  # Store preprocessed EEG for variability calculation
        baseline_variability = None  # Will be computed from Set 1
        
        set_labels = {
            1: 'Set 1: Baseline (No Distraction)',
            2: 'Set 2: Distraction + Response',
            3: 'Set 3: Distraction + No Response'
        }
        
        for set_id in [1, 2, 3]:
            if set_files[set_id] is None:
                print(f"\n--- {set_labels[set_id]} ---")
                print(f"No file found for this set, skipping...")
                continue
            
            print(f"\n--- {set_labels[set_id]} ---")
            
            eeg_data = analyzer.load_eeg_data(set_files[set_id])
            if eeg_data is None:
                print(f"Failed to load data")
                continue
            
            # Preprocess
            eeg_data = analyzer.preprocess_resting_eeg(eeg_data, remove_artifacts=True)
            
            # Store preprocessed data for baseline variability calculation
            set_eeg_data[set_id] = eeg_data
            
            # Analyze
            set_results[set_id] = analyzer.analyze_resting_state(eeg_data, set_labels[set_id])
            
            # Compute baseline variability from Set 1 (baseline)
            if set_id == 1:
                print(f"\n--- Computing Baseline Variability for Personalized Normalization ---")
                baseline_variability = analyzer.compute_baseline_variability(eeg_data, window_seconds=10)
        
        # ========================================
        # SET COMPARISONS
        # ========================================
        if len(set_results) >= 2:
            print(f"\n{'='*70}")
            print("SET COMPARISONS (Relative to Baseline)")
            print(f"{'='*70}")
            
            # Comparison 1: Baseline vs Set 2
            if 1 in set_results and 2 in set_results:
                print(f"\n{'='*60}")
                print("BASELINE vs SET 2 (Distraction + Response)")
                print(f"{'='*60}")
                comparison_1_2 = analyzer.compare_pre_post(
                    set_results[1], set_results[2], 
                    baseline_variability=baseline_variability
                )
            
            # Comparison 2: Baseline vs Set 3
            if 1 in set_results and 3 in set_results:
                print(f"\n{'='*60}")
                print("BASELINE vs SET 3 (Distraction + No Response)")
                print(f"{'='*60}")
                comparison_1_3 = analyzer.compare_pre_post(
                    set_results[1], set_results[3],
                    baseline_variability=baseline_variability
                )
            
            # Comparison 3: Set 2 vs Set 3
            if 2 in set_results and 3 in set_results:
                print(f"\n{'='*60}")
                print("SET 2 (Response Required) vs SET 3 (No Response)")
                print(f"{'='*60}")
                # For Set 2 vs Set 3 comparison, we can use Set 2's variability
                # But baseline variability is still more meaningful for context
                comparison_2_3 = analyzer.compare_pre_post(
                    set_results[2], set_results[3],
                    baseline_variability=baseline_variability
                )
            
            # Save visualization - name based on mode
            if mode == '1':
                file_prefix = "TaskEEG"
            else:  # mode == '3' (Combined)
                file_prefix = "Combined_TaskEEG"
            
            viz_path = os.path.join(base_path, f"{file_prefix}_Subject{subject_id}_L{level_id}T{task_id}_{timestamp}.png")
            
            # Create combined visualization
            if len(set_results) >= 2:
                fig, axes = plt.subplots(2, 3, figsize=(16, 10))
                fig.suptitle(f'EEG Analysis: Level {level_id} - {task_name} (Subject {subject_id})', fontsize=14, fontweight='bold')
                
                available_sets = sorted(set_results.keys())
                set_colors = {1: 'steelblue', 2: 'coral', 3: 'seagreen'}
                set_short_labels = {1: 'Baseline', 2: 'Set 2\n(Resp)', 3: 'Set 3\n(NoResp)'}
                
                # 1. Band Powers
                ax1 = axes[0, 0]
                bands = list(analyzer.frequency_bands.keys())
                x = np.arange(len(bands))
                width = 0.25
                
                for i, set_id in enumerate(available_sets):
                    powers = [safe_float(np.mean([set_results[set_id]['band_powers'][ch][b] 
                                      for ch in set_results[set_id]['band_powers']])) for b in bands]
                    ax1.bar(x + (i - 1) * width, powers, width, label=set_short_labels[set_id], 
                           color=set_colors[set_id], alpha=0.8)
                
                ax1.set_xlabel('Frequency Band')
                ax1.set_ylabel('Power (µV²)')
                ax1.set_title('Band Powers')
                ax1.set_xticks(x)
                ax1.set_xticklabels([b.capitalize() for b in bands], fontsize=8)
                ax1.legend(fontsize=8)
                
                # 2. Frequency Ratios
                ax2 = axes[0, 1]
                ratio_names = ['theta_beta', 'alpha_beta', 'theta_alpha']
                ratio_labels = ['θ/β\n(Attention)', 'α/β\n(Relaxation)', 'θ/α\n(Drowsiness)']
                x = np.arange(len(ratio_names))
                
                for i, set_id in enumerate(available_sets):
                    ratios = [safe_float(set_results[set_id]['ratios'][r]['average']) for r in ratio_names]
                    ax2.bar(x + (i - 1) * width, ratios, width, label=set_short_labels[set_id],
                           color=set_colors[set_id], alpha=0.8)
                
                ax2.set_xlabel('Ratio')
                ax2.set_ylabel('Value')
                ax2.set_title('Frequency Ratios')
                ax2.set_xticks(x)
                ax2.set_xticklabels(ratio_labels, fontsize=8)
                ax2.legend(fontsize=8)
                
                # 3. IAF
                ax3 = axes[0, 2]
                iafs = [safe_float(set_results[s]['iaf']['overall']) for s in available_sets]
                bars = ax3.bar([set_short_labels[s] for s in available_sets], iafs, 
                              color=[set_colors[s] for s in available_sets], alpha=0.8)
                ax3.set_ylabel('Frequency (Hz)')
                ax3.set_title('Individual Alpha Frequency')
                ax3.set_ylim(8, 12)
                for bar, val in zip(bars, iafs):
                    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                            f'{val:.2f}', ha='center', va='bottom', fontsize=9)
                
                # 4. FAA
                ax4 = axes[1, 0]
                faas = [safe_float(set_results[s]['faa']['faa']) for s in available_sets]
                bars = ax4.bar([set_short_labels[s] for s in available_sets], faas,
                              color=[set_colors[s] for s in available_sets], alpha=0.8)
                ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
                ax4.set_ylabel('FAA Value')
                ax4.set_title('Frontal Alpha Asymmetry')
                for bar, val in zip(bars, faas):
                    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
                
                # 5. Complexity
                ax5 = axes[1, 1]
                metrics = ['Sample\nEntropy', 'Higuchi\nFD']
                x = np.arange(len(metrics))
                
                for i, set_id in enumerate(available_sets):
                    values = [
                        safe_float(set_results[set_id]['complexity']['average']['sample_entropy']),
                        safe_float(set_results[set_id]['complexity']['average']['higuchi_fd'])
                    ]
                    ax5.bar(x + (i - 1) * width, values, width, label=set_short_labels[set_id],
                           color=set_colors[set_id], alpha=0.8)
                
                ax5.set_xlabel('Metric')
                ax5.set_ylabel('Value')
                ax5.set_title('Complexity Measures')
                ax5.set_xticks(x)
                ax5.set_xticklabels(metrics, fontsize=9)
                ax5.legend(fontsize=8)
                
                # 6. Fatigue comparison
                ax6 = axes[1, 2]
                fatigue_labels = []
                fatigue_values = []
                fatigue_colors = []
                
                if 1 in set_results and 2 in set_results:
                    fatigue_labels.append('Set 2\nvs Base')
                    fatigue_values.append(safe_float(comparison_1_2.get('fatigue_index', 0)))
                    fatigue_colors.append('coral')
                
                if 1 in set_results and 3 in set_results:
                    fatigue_labels.append('Set 3\nvs Base')
                    fatigue_values.append(safe_float(comparison_1_3.get('fatigue_index', 0)))
                    fatigue_colors.append('seagreen')
                
                if 2 in set_results and 3 in set_results:
                    fatigue_labels.append('Set 3\nvs Set 2')
                    fatigue_values.append(safe_float(comparison_2_3.get('fatigue_index', 0)))
                    fatigue_colors.append('purple')
                
                if fatigue_values:
                    bars = ax6.bar(fatigue_labels, fatigue_values, color=fatigue_colors, alpha=0.8)
                    ax6.set_ylabel('Fatigue Index')
                    ax6.set_title('Mental Fatigue Index')
                    ax6.set_ylim(0, 100)
                    ax6.axhline(y=50, color='red', linestyle='--', alpha=0.3, label='High')
                    ax6.axhline(y=25, color='orange', linestyle='--', alpha=0.3, label='Moderate')
                    
                    for bar, val in zip(bars, fatigue_values):
                        ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                                f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
                
                plt.tight_layout()
                try:
                    fig.savefig(viz_path, dpi=300, bbox_inches='tight', facecolor='white')
                    print(f"\n*** Visualization saved: {viz_path} ***")
                except Exception as viz_error:
                    print(f"\n*** Warning: Could not save visualization: {viz_error} ***")
                finally:
                    plt.close(fig)
                
                # Save text report for Task EEG analysis
                report_path = os.path.join(base_path, f"{file_prefix}_Subject{subject_id}_L{level_id}T{task_id}_{timestamp}.txt")
                try:
                    with open(report_path, 'w') as f:
                        f.write("="*80 + "\n")
                        f.write(f"TASK EEG ANALYSIS REPORT - SUBJECT {subject_id}\n")
                        f.write(f"Level {level_id} - {task_name}\n")
                        f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                        f.write("="*80 + "\n\n")
                        
                        # Set summaries
                        f.write("SET SUMMARIES\n")
                        f.write("-"*80 + "\n\n")
                        
                        for set_id in sorted(set_results.keys()):
                            results = set_results[set_id]
                            f.write(f"--- {set_labels[set_id]} ---\n")
                            f.write(f"Duration: {results['duration_seconds']:.1f} seconds\n")
                            f.write(f"Samples: {results['n_samples']}\n\n")
                            
                            f.write("Band Powers (average µV²):\n")
                            for band in analyzer.frequency_bands:
                                avg_power = np.mean([results['band_powers'][ch][band] 
                                                    for ch in results['band_powers']])
                                avg_relative = np.mean([results['relative_powers'][ch][band] 
                                                       for ch in results['relative_powers']])
                                f.write(f"  {band.capitalize():8s}: {avg_power:8.2f} ({avg_relative*100:5.1f}%)\n")
                            
                            f.write(f"\nFrequency Ratios:\n")
                            f.write(f"  Theta/Beta:  {results['ratios']['theta_beta']['average']:.3f}\n")
                            f.write(f"  Alpha/Beta:  {results['ratios']['alpha_beta']['average']:.3f}\n")
                            f.write(f"  Theta/Alpha: {results['ratios']['theta_alpha']['average']:.3f}\n")
                            
                            f.write(f"\nKey Metrics:\n")
                            f.write(f"  Individual Alpha Frequency: {results['iaf']['overall']:.2f} Hz\n")
                            f.write(f"  Frontal Alpha Asymmetry:    {results['faa']['faa']:.3f} ({results['faa']['interpretation']})\n")
                            f.write(f"  Sample Entropy:             {results['complexity']['average']['sample_entropy']:.3f}\n")
                            f.write(f"  Higuchi FD:                 {results['complexity']['average']['higuchi_fd']:.3f}\n")
                            f.write("\n")
                        
                        # Comparisons
                        f.write("\n" + "="*80 + "\n")
                        f.write("COMPARISONS\n")
                        f.write("="*80 + "\n\n")
                        
                        if 1 in set_results and 2 in set_results:
                            f.write("--- BASELINE vs SET 2 (Distraction + Response) ---\n")
                            f.write(f"Mental Fatigue Index: {comparison_1_2['fatigue_index']:.1f}/100\n")
                            f.write(f"Normalization: {comparison_1_2.get('normalization_type', 'heuristic')}\n")
                            if 'fatigue_details' in comparison_1_2:
                                f.write("Component Contributions:\n")
                                for detail in comparison_1_2['fatigue_details']:
                                    f.write(f"  {detail}\n")
                            f.write("\nInterpretations:\n")
                            for interp in comparison_1_2.get('interpretations', []):
                                f.write(f"  {interp}\n")
                            f.write("\n")
                        
                        if 1 in set_results and 3 in set_results:
                            f.write("--- BASELINE vs SET 3 (Distraction + No Response) ---\n")
                            f.write(f"Mental Fatigue Index: {comparison_1_3['fatigue_index']:.1f}/100\n")
                            f.write(f"Normalization: {comparison_1_3.get('normalization_type', 'heuristic')}\n")
                            if 'fatigue_details' in comparison_1_3:
                                f.write("Component Contributions:\n")
                                for detail in comparison_1_3['fatigue_details']:
                                    f.write(f"  {detail}\n")
                            f.write("\nInterpretations:\n")
                            for interp in comparison_1_3.get('interpretations', []):
                                f.write(f"  {interp}\n")
                            f.write("\n")
                        
                        if 2 in set_results and 3 in set_results:
                            f.write("--- SET 2 vs SET 3 ---\n")
                            f.write(f"Mental Fatigue Index: {comparison_2_3['fatigue_index']:.1f}/100\n")
                            f.write(f"Normalization: {comparison_2_3.get('normalization_type', 'heuristic')}\n")
                            if 'fatigue_details' in comparison_2_3:
                                f.write("Component Contributions:\n")
                                for detail in comparison_2_3['fatigue_details']:
                                    f.write(f"  {detail}\n")
                            f.write("\nInterpretations:\n")
                            for interp in comparison_2_3.get('interpretations', []):
                                f.write(f"  {interp}\n")
                            f.write("\n")
                        
                        # Baseline variability info
                        if baseline_variability:
                            f.write("\n" + "="*80 + "\n")
                            f.write("BASELINE VARIABILITY (for personalized normalization)\n")
                            f.write("="*80 + "\n")
                            f.write(f"Theta/Beta SD:     {baseline_variability.get('theta_beta_sd', 'N/A')}\n")
                            f.write(f"IAF SD:            {baseline_variability.get('iaf_sd', 'N/A')} Hz\n")
                            f.write(f"Alpha Relative SD: {baseline_variability.get('alpha_relative_sd', 'N/A')}%\n")
                            f.write(f"Sample Entropy SD: {baseline_variability.get('sample_entropy_sd', 'N/A')}\n")
                        
                        f.write("\n" + "="*80 + "\n")
                        f.write("END OF REPORT\n")
                        f.write("="*80 + "\n")
                    
                    print(f"*** Report saved: {report_path} ***")
                except Exception as report_error:
                    print(f"*** Warning: Could not save report: {report_error} ***")
        
        else:
            print("\nNeed at least 2 sets for comparison.")
    
    # ========================================
    # MODE 2 or 3: Resting State Analysis
    # ========================================
    if mode in ['2', '3']:
        print("\n" + "="*70)
        print("RESTING STATE ANALYSIS")
        print("="*70)
        
        if mode == '2':
            subject_id = input("\nEnter Subject ID: ").strip()
            if not subject_id:
                subject_id = "Unknown"
        
        # Try to auto-find resting state files
        resting_states_path = os.path.join(base_path, "RestingStates")
        pre_file = None
        post_file = None
        
        if os.path.exists(resting_states_path):
            print(f"\nSearching for resting state files in: {resting_states_path}")
            
            # Try multiple naming patterns
            patterns_to_try = [
                # Pattern 1: RestingStates/Subject{n}/RestingState_Results_xdf/subject{n}/restingstate_*.xdf
                (os.path.join(resting_states_path, f"Subject{subject_id}", "RestingState_Results_xdf", 
                              f"subject{subject_id}", f"restingstate_beg_sub{{{subject_id}}}.xdf"),
                 os.path.join(resting_states_path, f"Subject{subject_id}", "RestingState_Results_xdf", 
                              f"subject{subject_id}", f"restingstate_end_sub{{{subject_id}}}.xdf")),
                # Pattern 2: Without curly braces
                (os.path.join(resting_states_path, f"Subject{subject_id}", "RestingState_Results_xdf", 
                              f"subject{subject_id}", f"restingstate_beg_sub{subject_id}.xdf"),
                 os.path.join(resting_states_path, f"Subject{subject_id}", "RestingState_Results_xdf", 
                              f"subject{subject_id}", f"restingstate_end_sub{subject_id}.xdf")),
                # Pattern 3: Direct under Subject folder
                (os.path.join(resting_states_path, f"Subject{subject_id}", f"restingstate_beg.xdf"),
                 os.path.join(resting_states_path, f"Subject{subject_id}", f"restingstate_end.xdf")),
                # Pattern 4: CSV format
                (os.path.join(resting_states_path, f"Subject{subject_id}", f"restingstate_pre.csv"),
                 os.path.join(resting_states_path, f"Subject{subject_id}", f"restingstate_post.csv")),
            ]
            
            for pre_pattern, post_pattern in patterns_to_try:
                if os.path.exists(pre_pattern) and os.path.exists(post_pattern):
                    pre_file = pre_pattern
                    post_file = post_pattern
                    print(f"   Found pre-experiment:  {os.path.basename(pre_file)}")
                    print(f"   Found post-experiment: {os.path.basename(post_file)}")
                    break
            
            if pre_file is None:
                print("   Could not auto-find files. Please enter paths manually.")
        
        # If auto-find failed, ask for manual input
        if pre_file is None or post_file is None:
            print("\nEnter paths to resting state EEG files:")
            print("(Files can be in XDF or CSV format)")
            print()
            
            pre_file = input("Pre-experiment resting state file path: ").strip()
            post_file = input("Post-experiment resting state file path: ").strip()
        
        # Validate files exist
        if not os.path.exists(pre_file):
            print(f"Error: Pre-experiment file not found: {pre_file}")
            if mode == '2':
                return
        elif not os.path.exists(post_file):
            print(f"Error: Post-experiment file not found: {post_file}")
            if mode == '2':
                return
        else:
            # Load and process resting state data
            print("\n" + "="*70)
            print("LOADING RESTING STATE DATA")
            print("="*70)
            
            print("\nLoading pre-experiment recording...")
            pre_data = analyzer.load_eeg_data(pre_file)
            
            print("\nLoading post-experiment recording...")
            post_data = analyzer.load_eeg_data(post_file)
            
            if pre_data is not None and post_data is not None:
                # Preprocess
                print("\nPreprocessing...")
                pre_data = analyzer.preprocess_resting_eeg(pre_data)
                post_data = analyzer.preprocess_resting_eeg(post_data)
                
                # Analyze
                pre_results = analyzer.analyze_resting_state(pre_data, 'PRE-EXPERIMENT')
                post_results = analyzer.analyze_resting_state(post_data, 'POST-EXPERIMENT')
                
                # Compute baseline variability from pre-experiment data
                print(f"\n--- Computing Baseline Variability for Personalized Normalization ---")
                baseline_variability = analyzer.compute_baseline_variability(pre_data, window_seconds=10)
                
                # Compare with personalized normalization
                comparison = analyzer.compare_pre_post(pre_results, post_results, 
                                                       baseline_variability=baseline_variability)
                
                # Save outputs - name based on mode
                if mode == '2':
                    resting_prefix = "RestingState"
                else:  # mode == '3' (Combined)
                    resting_prefix = "Combined_RestingState"
                
                viz_path = os.path.join(base_path, f"{resting_prefix}_Subject{subject_id}_{timestamp}.png")
                analyzer.visualize_comparison(pre_results, post_results, comparison, save_path=viz_path)
                
                report_path = os.path.join(base_path, f"{resting_prefix}_Subject{subject_id}_{timestamp}.txt")
                analyzer.generate_report(pre_results, post_results, comparison, subject_id, save_path=report_path)
                
                print(f"\n*** Resting state outputs saved ***")
                print(f"    Visualization: {viz_path}")
                print(f"    Report: {report_path}")
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE")
    print("="*70)


In [ ]:
# Run this cell to execute the script
main()
